In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import bacco
import h5py

%load_ext autoreload
%autoreload 2

### Quick Introduction

These are resimulations of the MillenniumTNG hydrodynamical simulation of galaxy formation.

This is a cosmological simulation run on a periodic box with L=500[Mpc/h], and usual Planck cosmology using the IllustrisTNG galaxy formation model, and the astrophysical and feedback parameters basically equal to those used in running the TNG100 and 300 simulations.

The MillenniumTNG simulation has 4 types of particles:
- Type0 -> Gas (Varying mass Voronoi cells)
- Type1 -> Dark Matter (Fixed Mass Particles)
- Type4 -> Stars (Varying Mass Particles)
- Type5 -> Black-Holes (Varying Mass Particles)

As for the resimulations of the MillenniumTNG, they have 6 types of particles:
- Type0 -> Gas (Varying mass Voronoi cells)
- Type1 -> High-Resolution Dark Matter (Fixed Mass Particles)
- Type2 -> Intermedite-Resolution Dark Matter (Varying Mass Particles)
- Type3 -> Low-Resolution Dark Matter (Fixed Mass Particles)
- Type4 -> Stars (Varying Mass Particles)
- Type5 -> Black-Holes (Varying Mass Particles)


### Structures

In these simulations, subhalos that have stellar mass can be typically thought of as galaxies, although some caveats may apply.

### Using Bacco

Here I will load simply the structure catalogs (halos and subhalos), not the particles. This can also be done, simply by passing a property to the Simulation class called dm_file, where you specify what is the location of the snapshot files.

In [ ]:
sigma8 = 0.8159 #CHECK ME
ns     = 0.9667 #CHECK ME
tau    = 0.0965 #CHECK ME

# SNAP LIST
snaps = [264, 214, 200, 100]
zoom = {}

# NAME LIST
name_list = ['LH_'+str(i) for i in range(14,16)]

for i in range(len(name_list)):
    zoom[name_list[i]] = {}
    for snap in snaps:
        try:
            base = "/cosmos_storage/simulations/TNG_Family/MN5_resims/"+name_list[i]+"/hydro_output/"
            zoom[name_list[i]][snap] = bacco.Simulation(basedir=base, halo_file="groups_{:03d}/fof_subhalo_tab_{:03d}".format(snap,snap), sim_format='TNG500', fixedPk=True, use_orphans=False,\
                                    tau=tau, ns=ns, sigma8=sigma8, tree_file="groups_{:03d}/subhalo_prog_{:03d}".format(snap,snap), use_ids=True, numpart=4320)
        except:
           print('failed to load', name_list[i], snap)

### Using h5py

In [ ]:
dm = {}
gas = {}
stars = {}

basePath = "/cosmos_storage/simulations/TNG_Family/MN5_resims/LH_13/hydro_output/"

# Snapshot file -- information about particles
with h5py.File(basePath+"/snapdir_264/snapshot_264.0.hdf5", 'r') as file:
    # High-res DM
    dm['ids'] = np.array(file['PartType1']['ParticleIDs'], dtype=np.uint64)
    dm['pos'] = file['PartType1']['Coordinates'][...]

    # Gas
    gas['pos'] = file['PartType0']['Coordinates'][...]
    gas['mass'] = file['PartType0']['Masses'][...]

    # Stars
    stars['pos'] = file['PartType4']['Coordinates'][...]
    stars['mass'] = file['PartType4']['Masses'][...]
        
    print(file['PartType0'].keys())

# Group-File -- Information about the halos and subhalos
with h5py.File(basePath+"/groups_264/fof_subhalo_tab_264.0.hdf5", 'r') as file:
    # Number of particles of each type for each subhalo (galaxy)
    lenType = file['Subhalo']['SubhaloLenType'][...]


In [ ]:
dm['offsets'] = np.cumsum( np.hstack(([0], lenType[:,1])) )
gas['offsets'] = np.cumsum( np.hstack(([0], lenType[:,0])) )
stars['offsets'] = np.cumsum( np.hstack(([0], lenType[:,4])) )

In [ ]:
# Load particles in the halo of index isub
isub = 0

dm_pos = dm['pos'][dm['offsets'][isub]:dm['offsets'][isub+1],:]
dm_mass = np.ones(dm_pos.shape[0])

star_pos = stars['pos'][stars['offsets'][isub]:stars['offsets'][isub+1],:]
star_mass = stars['mass'][stars['offsets'][isub]:stars['offsets'][isub+1]]

gas_pos = gas['pos'][gas['offsets'][isub]:gas['offsets'][isub+1],:]
gas_mass = gas['mass'][gas['offsets'][isub]:gas['offsets'][isub+1]]

In [ ]:
grid_DM = np.histogram2d(dm_pos[:,0], dm_pos[:,1],bins=(np.linspace(86,92,100), np.linspace(317,323,100)))
grid_gas = np.histogram2d(gas_pos[:,0], gas_pos[:,1],bins=(np.linspace(86,92,100), np.linspace(317,323,100)))
grid_stars = np.histogram2d(star_pos[:,0], star_pos[:,1],bins=(np.linspace(86,92,100), np.linspace(317,323,100)))

In [ ]:
fig, ax = plt.subplots(1,3,dpi=200)

ax[0].imshow(np.log10(1+grid_DM[0]))
ax[1].imshow(np.log10(1+grid_gas[0]))
ax[2].imshow(np.log10(1+grid_stars[0]))